# Basic

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import numpy as np
import tqdm
import networkx as nx


import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer
import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct


id_to_label, combined_subgraphs = graph_fct.get_combined_combined_subgraphs_and_id2label()
df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = df_mapped["id"].unique().to_list()


In [ ]:
D = config.TokenizerParam().max_dist_candidate
lambdas_list = np.round(np.arange(0.1, 1.1, 0.2), 1)

for lam in tqdm.tqdm(lambdas_list):
    selector = tokenizer.LazyGreedyTokenSelector(combined_subgraphs, mapped_ids, D = D, lam = lam)
    history = selector.select(k = int(len(mapped_ids)/2),n_jobs=-1)   # [(token, marginal_gain, cumulative_score), ...]
    df_history =(
        pl.DataFrame(
        history, schema=["token", "gain", "cumulative_score"], orient="row")
        .with_columns(
        pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
        .with_row_index())
    df_history.write_parquet(f"{config.CandidateLists().path_greedy_tree}{lam}.parquet")

  0%|          | 0/5 [00:00<?, ?it/s]

  seeding candidates: 0/75964  (16 workers)
  seeding candidates: 5000/75964  (16 workers)
  seeding candidates: 10000/75964  (16 workers)
  seeding candidates: 15000/75964  (16 workers)
  seeding candidates: 20000/75964  (16 workers)
  seeding candidates: 25000/75964  (16 workers)
  seeding candidates: 30000/75964  (16 workers)
  seeding candidates: 35000/75964  (16 workers)
  seeding candidates: 40000/75964  (16 workers)
  seeding candidates: 45000/75964  (16 workers)
  seeding candidates: 50000/75964  (16 workers)
  seeding candidates: 55000/75964  (16 workers)
  seeding candidates: 60000/75964  (16 workers)
  seeding candidates: 65000/75964  (16 workers)
  seeding candidates: 70000/75964  (16 workers)
  seeding candidates: 75000/75964  (16 workers)
  seeding candidates: 75964/75964  (16 workers)
[   1/23075] token='129265001'     gain=0.00629  score=0.00629  opt<=0.00995  (re-evals=1, queue=75963)
[   2/23075] token='410607006'     gain=0.00279  score=0.00908  opt<=0.01437  (re-eva